# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Display the dataset's name and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, their fields, columns, and associated `@id`s.

In [ ]:
# List all record sets in the dataset
record_sets = [rs for rs in dataset.metadata.record_sets]
print("Record sets and their @id's:")
for rs in record_sets:
    print(f"- RecordSet: {rs['@id']} (name: {rs.get('name', '<none>')})")
    print("  Fields in this record set:")
    for field in rs.get('field', []):
        # Each field is a dict or an ID; get the ID either way
        if isinstance(field, dict):
            field_id = field.get('@id', str(field))
            field_name = field.get('name', '<none>')
        else:
            field_id = str(field)
            field_name = '<none>'
        print(f"    - {field_id} (name: {field_name})")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Choose the main survey responses record set (assuming it's the first for demo):
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
# List of all loaded DataFrames for each record set
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set: {record_set_id} with shape {df.shape}")
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {e}")

# Select the primary survey record set for further analysis (edit as appropriate)
primary_record_set = record_set_ids[0] if record_set_ids else None
if primary_record_set:
    print(f"\nColumns for record set {primary_record_set}:")
    print(dataframes[primary_record_set].columns.tolist())
    display(dataframes[primary_record_set].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.


In [ ]:
# Select a numeric field for analysis
# You will find the correct field names/ids above; adjust as appropriate
df = dataframes[primary_record_set]
# Example: choose GAD_7_score, PHQ_9_score, or PSQ_score if present
for candidate_field in ['GAD_7_score', 'PHQ_9_score', 'PSQ_score', 'gad7_score', 'phq9_score', 'psq_score']:
    if candidate_field in df.columns:
        numeric_field = candidate_field
        break
else:
    numeric_field = df.select_dtypes('number').columns[0] if len(df.select_dtypes('number').columns) else df.columns[0]

print(f"Using numeric field: {numeric_field}")

# Filter records where field is not null and greater than a threshold
threshold = 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}: {len(filtered_df)} found.")
display(filtered_df.head())

# Normalize the selected numeric field
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to group by a demographic field (e.g., 'Gender', 'gender', 'Age', 'age', etc.)
group_candidates = ['Gender', 'gender', 'Age', 'age', 'level_of_education', 'marital_status']
for group_field in group_candidates:
    if group_field in df.columns:
        break
else:
    group_field = None

if group_field is not None:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped data by {group_field}: Mean of {numeric_field}")
    display(grouped_df)
else:
    print("No grouping field found for demonstration.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if group_field is not None:
    plt.figure(figsize=(8, 5))
    sns.barplot(x=group_field, y=numeric_field, data=filtered_df, ci=None)
    plt.title(f'Mean {numeric_field} by {group_field} (Filtered, > {threshold})')
    plt.ylabel(f"Mean {numeric_field}")
    plt.xlabel(group_field)
    plt.show()

plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field].dropna(), kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load and explore the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library. We:

- Loaded the dataset and reviewed its metadata and available record sets and fields by their `@id`.
- Extracted survey data and visualized the distribution of a key mental health metric (e.g., GAD-7 score) across demographic groups.
- Demonstrated basic filtering, normalization, and grouping for exploratory data analysis.

**Next steps could include statistical testing, advanced modeling, or more granular analysis of relationships across multiple fields.**